# Alphabet Volatility Regimes with Extended Validation

An alternate GOOGL regime model with walk-forward validation.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
================================================================================
ENHANCED THRESHOLD REGIME TRADING STRATEGY - VERSION 2.3 XGBOOST (FIXED)
================================================================================

FIXES IN THIS VERSION:
---------------------
1. XGBoost predict() no longer filters rows with ANY missing features
   - XGBoost natively handles NaN, so we pass data directly
   - Only skip rows where ALL features are missing
2. Proper lagging applied BEFORE predictions in walk-forward
3. DataFrame updates use .loc properly to avoid view/copy issues

CRITICAL TEMPORAL FLOW:
----------------------
Training: Features[T-1] → Target[T+5 realized vol]
Prediction: Features[T-1] → Forecast[T+5 vol] → Use directly for regime
Execution: Signal[T-1] → Execute at Open[T]

================================================================================
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import warnings
from scipy import stats
from itertools import product
import json

try:
    from arch import arch_model

    ARCH_AVAILABLE = True
except ImportError:
    print("[WARNING] arch package not available. GARCH features will be disabled.")
    ARCH_AVAILABLE = False
try:
    import xgboost as xgb

    XGBOOST_AVAILABLE = True
except ImportError:
    print("[WARNING] XGBoost not available. Will fallback to Yang-Zhang volatility.")
    XGBOOST_AVAILABLE = False
warnings.filterwarnings("ignore")
TICKER = "GOOGL"
USE_XGBOOST = True


@dataclass
class StrategyConfig:
    """Configuration for Enhanced Threshold Regime Strategy V2.3."""

    ticker: str = TICKER
    benchmark_ticker: str = "URTH"
    start_date: str = "2019-01-01"
    end_date: str = "2025-06-22"
    warmup_buffer_days: int = 200
    validate_feature_warmup: bool = True
    vol_lookback: int = 20
    quantile_window: int = 252
    n_regimes: int = 2
    regime_quantile: float = 0.5
    yz_k_numerator: float = 0.34
    yz_k_denominator_constant: float = 1.34
    yz_annualization_factor: int = 252
    rsi_period: int = 14
    rsi_oversold: float = 30.0
    rsi_overbought: float = 70.0
    har_windows: List[int] = field(default_factory=lambda: [1, 5, 21, 63])
    garch_p: int = 1
    garch_o: int = 1
    garch_q: int = 1
    garch_min_obs: int = 252
    garch_forecast_horizon: int = 1
    bpv_window: int = 21
    momentum_windows: List[int] = field(default_factory=lambda: [5, 21, 63])
    ema_fast: int = 12
    ema_slow: int = 26
    kurtosis_window: int = 63
    hurst_window: int = 126
    hurst_lags: range = field(default_factory=lambda: range(2, 20))
    use_xgboost: bool = USE_XGBOOST
    xgb_forecast_horizon: int = 5
    xgb_min_train_samples: int = 252
    xgb_retrain_frequency: int = 21
    xgb_max_depth: int = 4
    xgb_learning_rate: float = 0.05
    xgb_n_estimators: int = 100
    xgb_subsample: float = 0.8
    xgb_colsample_bytree: float = 0.8
    xgb_early_stopping_rounds: int = 20
    xgb_validation_split: float = 0.2
    persistence_days: int = 3
    use_persistence_filter: bool = True
    initial_capital: float = 100000.0
    compounding_mode: bool = True
    leverage: float = 1.0
    low_vol_allocation: float = 1.0
    high_vol_allocation: float = 0.0
    commission_per_share: float = 0.0035
    commission_min: float = 0.35
    commission_max_pct: float = 0.01
    spread_bps: float = 2.0
    market_impact_bps: float = 3.0
    margin_rate_annual: float = 0.06
    min_train_window: int = 504
    validation_window: int = 126
    refit_frequency: int = 126
    step_size: int = 63
    vol_lookback_grid: List[int] = field(default_factory=lambda: [10, 15, 20, 30, 42, 63])
    quantile_window_grid: List[int] = field(default_factory=lambda: [126, 189, 252])
    regime_quantile_grid: List[float] = field(default_factory=lambda: [0.3, 0.4, 0.5, 0.6, 0.7])
    persistence_days_grid: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 5])
    fast_mode: bool = False
    optimization_objective: str = "sharpe"
    max_param_combinations: int = 200
    gri_threshold: float = 0.65
    min_psr: float = 0.95
    min_wfe: float = 0.5
    num_trials: int = 100
    psr_benchmark_sharpe: float = 0.0


class YahooDataFetcher:
    """Fetch historical OHLCV data from Yahoo Finance."""

    def __init__(self):
        pass

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """Fetch historical OHLCV data. Returns DataFrame with Adj Close as primary price."""
        print(f"[DATA] Fetching {ticker} from {start_date} to {end_date}...")
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
        except RuntimeError as e:
            raise ValueError(f"API request failed for {ticker}: {str(e)}")
        if "historical" not in data:
            raise ValueError(f"No historical data found for {ticker}")
        historical = data["historical"]
        if len(historical) == 0:
            raise ValueError(f"Empty historical data for {ticker}")
        df = pd.DataFrame(historical)
        df["date"] = pd.to_datetime(df["date"])
        df.set_index("date", inplace=True)
        df.sort_index(inplace=True)
        column_mapping = {
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "adjClose": "Adj Close",
            "volume": "Volume",
        }
        available_cols = {k: v for k, v in column_mapping.items() if k in df.columns}
        df = df[list(available_cols.keys())].rename(columns=available_cols)
        if "Adj Close" in df.columns and "Close" in df.columns:
            adj_ratio = df["Adj Close"] / df["Close"]
            df["Open"] = df["Open"] * adj_ratio
            df["High"] = df["High"] * adj_ratio
            df["Low"] = df["Low"] * adj_ratio
            df["Close"] = df["Adj Close"]
        print(f"[DATA] Retrieved {len(df)} trading days for {ticker}")
        return df


class XGBoostVolatilityPredictor:
    """
    XGBoost model to predict 5-day forward realized volatility.

    FIX: predict() now properly handles NaN values (XGBoost native capability).
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.model = None
        self.feature_columns = None
        self.last_train_date = None
        self.training_history = []

    def prepare_training_data(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        """
        Prepare training data with proper temporal alignment.

        FIX: Explicitly exclude XGBoost outputs from feature set.
        """
        forward_rv_squared = pd.Series(index=df.index, dtype=float)
        for i in range(len(df) - self.config.xgb_forecast_horizon):
            idx = df.index[i]
            future_returns = df["log_returns"].iloc[
                i + 1 : i + 1 + self.config.xgb_forecast_horizon
            ]
            if len(future_returns) == self.config.xgb_forecast_horizon:
                rv_squared = (future_returns**2).sum()
                forward_rv_squared.loc[idx] = rv_squared
        y = np.sqrt(forward_rv_squared * (252 / self.config.xgb_forecast_horizon))
        exclude_patterns = ["xgb_vol", "target", "signal", "regime", "volatility"]
        feature_cols = [
            col
            for col in df.columns
            if col.endswith("_lagged")
            and (not any((pattern in col.lower() for pattern in exclude_patterns)))
        ]
        X = df[feature_cols].copy()
        self.feature_columns = feature_cols
        valid_mask = y.notna()
        X_clean = X[valid_mask]
        y_clean = y[valid_mask]
        print(f"    [XGB] Training data: {len(X_clean)} samples, {len(feature_cols)} features")
        print(f"    [XGB] Excluded patterns: {exclude_patterns}")
        return (X_clean, y_clean)

    def train(self, df: pd.DataFrame, verbose: bool = False) -> Dict[str, float]:
        """Train XGBoost model on provided data."""
        if not XGBOOST_AVAILABLE:
            return {"error": "XGBoost not available"}
        X, y = self.prepare_training_data(df)
        if len(X) < self.config.xgb_min_train_samples:
            if verbose:
                print(
                    f"    [XGB] Insufficient samples: {len(X)} < {self.config.xgb_min_train_samples}"
                )
            return {"error": "Insufficient data", "n_samples": len(X)}
        split_idx = int(len(X) * (1 - self.config.xgb_validation_split))
        X_train, X_val = (X.iloc[:split_idx], X.iloc[split_idx:])
        y_train, y_val = (y.iloc[:split_idx], y.iloc[split_idx:])
        if verbose:
            print(f"    [XGB] Train: {len(X_train)}, Val: {len(X_val)}")
        params = {
            "objective": "reg:squarederror",
            "max_depth": self.config.xgb_max_depth,
            "learning_rate": self.config.xgb_learning_rate,
            "n_estimators": self.config.xgb_n_estimators,
            "subsample": self.config.xgb_subsample,
            "colsample_bytree": self.config.xgb_colsample_bytree,
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": 0,
            "early_stopping_rounds": self.config.xgb_early_stopping_rounds,
        }
        self.model = xgb.XGBRegressor(**params)
        self.model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        train_pred = self.model.predict(X_train)
        val_pred = self.model.predict(X_val)
        train_rmse = np.sqrt(np.mean((y_train - train_pred) ** 2))
        val_rmse = np.sqrt(np.mean((y_val - val_pred) ** 2))
        train_r2 = 1 - np.sum((y_train - train_pred) ** 2) / np.sum((y_train - y_train.mean()) ** 2)
        val_r2 = 1 - np.sum((y_val - val_pred) ** 2) / np.sum((y_val - y_val.mean()) ** 2)
        metrics = {
            "train_rmse": train_rmse,
            "val_rmse": val_rmse,
            "train_r2": train_r2,
            "val_r2": val_r2,
            "n_train": len(X_train),
            "n_val": len(X_val),
            "n_features": len(self.feature_columns),
        }
        if verbose:
            print(f"    [XGB] Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}")
            print(f"    [XGB] Train R²: {train_r2:.4f}, Val R²: {val_r2:.4f}")
        self.last_train_date = df.index[-1]
        self.training_history.append({"date": self.last_train_date, **metrics})
        return metrics

    def predict(self, df: pd.DataFrame) -> pd.Series:
        """
        Generate volatility forecasts.

        CRITICAL FIX: XGBoost natively handles NaN values in features.
        We only skip rows where ALL features are missing (impossible case).
        Previously filtered ANY missing feature → 0 predictions.
        Now passes data directly to XGBoost → proper predictions.
        """
        if self.model is None or self.feature_columns is None:
            print("[XGB] Model not trained, returning NaN")
            return pd.Series(np.nan, index=df.index)
        if not XGBOOST_AVAILABLE:
            return pd.Series(np.nan, index=df.index)
        X = df[self.feature_columns].copy()
        missing_cols = set(self.feature_columns) - set(X.columns)
        if missing_cols:
            print(f"[XGB WARNING] Missing feature columns: {missing_cols}")
            for col in missing_cols:
                X[col] = np.nan
        valid_mask = X.notna().any(axis=1)
        if valid_mask.sum() == 0:
            print(f"[XGB WARNING] No valid rows for prediction")
            return pd.Series(np.nan, index=df.index)
        predictions = pd.Series(np.nan, index=df.index)
        X_valid = X[valid_mask]
        try:
            preds = self.model.predict(X_valid)
            predictions.loc[valid_mask] = preds
        except Exception as e:
            print(f"[XGB ERROR] Prediction failed: {e}")
            return pd.Series(np.nan, index=df.index)
        return predictions

    def get_feature_importance(self, top_n: int = 20) -> pd.DataFrame:
        """Get feature importance from trained model."""
        if self.model is None or self.feature_columns is None:
            return pd.DataFrame()
        importance = pd.DataFrame(
            {"feature": self.feature_columns, "importance": self.model.feature_importances_}
        ).sort_values("importance", ascending=False)
        return importance.head(top_n)


class EnhancedFeatureEngineer:
    """Feature engineering for XGBoost training - LEAKAGE FIXED."""

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.xgb_predictor = XGBoostVolatilityPredictor(config) if config.use_xgboost else None

    def engineer_features(self, df: pd.DataFrame, train_xgb: bool = False) -> pd.DataFrame:
        """
        Main feature engineering pipeline.

        CRITICAL FIX: Prevent XGBoost output from becoming input feature.
        FIX V2.4: Remove double-lagging of XGBoost forecasts.
        """
        print("\n[FEATURE ENGINEERING] Creating features for XGBoost...")
        df = df.copy()
        xgb_output_cols = [c for c in df.columns if "xgb_vol" in c.lower()]
        if xgb_output_cols:
            print(f"  [CLEANUP] Dropping previous XGBoost outputs: {xgb_output_cols}")
            df.drop(columns=xgb_output_cols, inplace=True, errors="ignore")
        df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))
        df["returns"] = df["Close"].pct_change()
        print(f"  [1/9] Returns calculated")
        df = self._calculate_volatility_estimators(df)
        print(f"  [2/9] Volatility estimators calculated")
        df = self._calculate_volatility_decomposition(df)
        print(f"  [3/9] Volatility decomposition calculated")
        df = self._calculate_har_features(df)
        print(f"  [4/9] HAR features calculated")
        if ARCH_AVAILABLE:
            df = self._calculate_garch_forecast(df)
            print(f"  [5/9] GARCH forecasts calculated")
        else:
            df["garch_forecast"] = np.nan
            print(f"  [5/9] GARCH skipped (arch not available)")
        df = self._calculate_momentum_features(df)
        print(f"  [6/9] Momentum features calculated")
        df = self._calculate_microstructure_features(df)
        print(f"  [7/9] Microstructure features calculated")
        exclude_from_lagging = {
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",
            "returns",
            "log_returns",
            "target_allocation",
            "signal",
            "regime",
            "regime_name",
            "xgb_vol_forecast",
            "volatility",
            "vol_threshold",
            "regime_raw",
            "regime_duration",
        }
        candidate_features = [
            col
            for col in df.columns
            if col not in exclude_from_lagging and (not col.endswith("_lagged"))
        ]
        for col in candidate_features:
            df[f"{col}_lagged"] = df[col].shift(1)
        print(f"  [8/9] Feature lagging applied to {len(candidate_features)} features")
        if self.config.use_xgboost and XGBOOST_AVAILABLE:
            if train_xgb:
                print(f"  [9/9] Training XGBoost model...")
                train_metrics = self.xgb_predictor.train(df, verbose=True)
            xgb_forecasts = self.xgb_predictor.predict(df)
            df["xgb_vol_forecast"] = xgb_forecasts
            valid_preds = xgb_forecasts.notna().sum()
            print(f"  [9/9] XGBoost forecasts: {valid_preds} / {len(df)} (uses T-1 features)")
        else:
            df["xgb_vol_forecast"] = np.nan
            print(f"  [9/9] XGBoost skipped - Yang-Zhang fallback")
        self._report_nan_stats(df)
        return df

    def _calculate_volatility_estimators(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate volatility estimators for XGBoost features."""
        window = self.config.vol_lookback
        log_open = np.log(df["Open"])
        log_high = np.log(df["High"])
        log_low = np.log(df["Low"])
        log_close = np.log(df["Close"])
        log_close_prev = log_close.shift(1)
        overnight_ret = log_open - log_close_prev
        var_overnight = overnight_ret.rolling(window).var()
        open_close_ret = log_close - log_open
        var_open_close = open_close_ret.rolling(window).var()
        rs_component = (log_high - log_close) * (log_high - log_open) + (log_low - log_close) * (
            log_low - log_open
        )
        var_rs = rs_component.rolling(window).mean()
        k = self.config.yz_k_numerator / (
            self.config.yz_k_denominator_constant + (window + 1) / (window - 1)
        )
        yz_variance = var_overnight + k * var_open_close + (1 - k) * var_rs
        df["yang_zhang_vol"] = np.sqrt(yz_variance) * np.sqrt(self.config.yz_annualization_factor)
        log_hl = np.log(df["High"] / df["Low"])
        log_co = np.log(df["Close"] / df["Open"])
        gk_daily = np.sqrt(0.5 * log_hl**2 - (2 * np.log(2) - 1) * log_co**2)
        df["garman_klass_vol"] = gk_daily.rolling(window).mean() * np.sqrt(
            self.config.yz_annualization_factor
        )
        df["realized_vol"] = df["log_returns"].rolling(window).std() * np.sqrt(
            self.config.yz_annualization_factor
        )
        return df

    def _calculate_volatility_decomposition(self, df: pd.DataFrame) -> pd.DataFrame:
        """Decompose volatility into continuous and jump components."""
        window = self.config.bpv_window
        abs_returns = df["log_returns"].abs()
        mu_1 = np.sqrt(2 / np.pi)
        bpv_raw = abs_returns * abs_returns.shift(1)
        bpv = bpv_raw.rolling(window).sum() * mu_1 ** (-2)
        rv = (df["log_returns"] ** 2).rolling(window).sum()
        jump_raw = rv - bpv
        jump = np.maximum(jump_raw, 0)
        df["bipower_variation"] = bpv
        df["realized_variance"] = rv
        df["jump_component"] = jump
        df["jump_ratio"] = jump / (rv + 1e-09)
        return df

    def _calculate_har_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate HAR features."""
        rv_daily = df["log_returns"] ** 2
        for window in self.config.har_windows:
            har_rv = rv_daily.rolling(window).mean()
            har_vol = np.sqrt(har_rv * self.config.yz_annualization_factor)
            har_log = np.log(har_vol + 1e-09)
            window_name = self._get_window_name(window)
            df[f"har_rv_{window_name}"] = har_rv
            df[f"har_vol_{window_name}"] = har_vol
            df[f"har_log_{window_name}"] = har_log
        return df

    def _get_window_name(self, days: int) -> str:
        """Convert window size to descriptive name."""
        if days == 1:
            return "daily"
        elif days <= 7:
            return "weekly"
        elif days <= 30:
            return "monthly"
        elif days <= 90:
            return "quarterly"
        else:
            return f"{days}d"

    def _calculate_garch_forecast(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate EGARCH forecasts."""
        if not ARCH_AVAILABLE:
            df["garch_forecast"] = np.nan
            return df
        print("    Calculating GARCH forecasts...")
        returns = df["log_returns"].dropna() * 100
        min_obs = self.config.garch_min_obs
        if len(returns) < min_obs:
            df["garch_forecast"] = np.nan
            return df
        forecasts = pd.Series(index=df.index, dtype=float)
        refit_freq = 21
        last_fit_idx = 0
        last_model = None
        for i in range(min_obs, len(returns)):
            current_date = returns.index[i]
            if i - last_fit_idx >= refit_freq or last_model is None:
                train_returns = returns.iloc[max(0, i - min_obs) : i]
                try:
                    model = arch_model(
                        train_returns,
                        mean="Constant",
                        vol="EGARCH",
                        p=self.config.garch_p,
                        o=self.config.garch_o,
                        q=self.config.garch_q,
                        dist="skewt",
                    )
                    res = model.fit(disp="off", show_warning=False, options={"maxiter": 100})
                    last_model = res
                    last_fit_idx = i
                except:
                    if last_model is None:
                        continue
            if last_model is not None:
                try:
                    forecast = last_model.forecast(horizon=1, reindex=False)
                    forecast_var = forecast.variance.values[-1][0]
                    forecast_var_decimal = (
                        forecast_var / 10000 * self.config.yz_annualization_factor
                    )
                    forecast_vol = np.sqrt(forecast_var_decimal)
                    forecast_vol = np.clip(forecast_vol, 0, 2.0)
                    forecasts.loc[current_date] = forecast_vol
                except:
                    pass
        df["garch_forecast"] = forecasts
        return df

    def _calculate_momentum_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate momentum indicators."""
        df["rsi"] = self._calculate_rsi(df["Close"], self.config.rsi_period)
        df["rsi_extreme"] = (
            (df["rsi"] < self.config.rsi_oversold) | (df["rsi"] > self.config.rsi_overbought)
        ).astype(float)
        df["ema_fast"] = df["Close"].ewm(span=self.config.ema_fast, adjust=False).mean()
        df["ema_slow"] = df["Close"].ewm(span=self.config.ema_slow, adjust=False).mean()
        df["ema_crossover"] = (df["ema_fast"] - df["ema_slow"]) / df["ema_slow"]
        for window in self.config.momentum_windows:
            momentum = df["Close"].pct_change(window)
            df[f"momentum_{window}d"] = momentum
        for window in [20, 50, 200]:
            ma = df["Close"].rolling(window).mean()
            df[f"dist_ma{window}"] = (df["Close"] - ma) / ma
        return df

    def _calculate_rsi(self, series: pd.Series, periods: int) -> pd.Series:
        """Calculate RSI."""
        delta = series.diff()
        gain = delta.where(delta > 0, 0).ewm(alpha=1 / periods, adjust=False).mean()
        loss = (-delta.where(delta < 0, 0)).ewm(alpha=1 / periods, adjust=False).mean()
        rs = gain / (loss + 1e-09)
        rsi = 100 - 100 / (1 + rs)
        return rsi

    def _calculate_microstructure_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calculate microstructure features."""
        df["hl_spread"] = (df["High"] - df["Low"]) / df["Close"]
        df["hl_spread_ma"] = df["hl_spread"].rolling(21).mean()
        df["hl_spread_std"] = df["hl_spread"].rolling(21).std()
        df["hl_spread_zscore"] = (df["hl_spread"] - df["hl_spread_ma"]) / (
            df["hl_spread_std"] + 1e-09
        )
        df["volume_ma"] = df["Volume"].rolling(21).mean()
        df["volume_std"] = df["Volume"].rolling(21).std()
        df["volume_shock"] = (df["Volume"] - df["volume_ma"]) / (df["volume_std"] + 1e-09)
        df["kurtosis"] = df["log_returns"].rolling(self.config.kurtosis_window).kurt()
        df["hurst"] = (
            df["log_returns"]
            .rolling(self.config.hurst_window)
            .apply(lambda x: self._hurst_exponent(x.values), raw=False)
        )
        df["autocorr_1"] = (
            df["log_returns"]
            .rolling(63)
            .apply(lambda x: x.autocorr(lag=1) if len(x) > 1 else 0, raw=False)
        )
        df["downside_dev"] = df["log_returns"].clip(upper=0).rolling(21).std() * np.sqrt(252)
        return df

    def _hurst_exponent(self, ts: np.ndarray) -> float:
        """Calculate Hurst exponent."""
        n = len(ts)
        if n < 50:
            return np.nan
        lags = range(2, min(20, n // 4))
        tau = []
        for lag in lags:
            std_dev = np.std(ts[lag:] - ts[:-lag])
            if std_dev > 0:
                tau.append(std_dev)
        if len(tau) < 2:
            return np.nan
        try:
            poly = np.polyfit(np.log(list(lags)[: len(tau)]), np.log(tau), 1)
            hurst = poly[0]
            return max(0, min(1, hurst))
        except:
            return np.nan

    def _report_nan_stats(self, df: pd.DataFrame):
        """Report NaN statistics."""
        print("\n  NaN Statistics:")
        print("  " + "-" * 60)
        feature_cols = [col for col in df.columns if col.endswith("_lagged")]
        total_rows = len(df)
        for col in feature_cols[:10]:
            nan_count = df[col].isna().sum()
            pct_nan = nan_count / total_rows * 100
            print(f"  {col:40s}: {nan_count:5d} NaN ({pct_nan:5.1f}%)")
        print("  " + "-" * 60)


def validate_feature_readiness(df: pd.DataFrame, backtest_start: str, config: StrategyConfig):
    """
    Validate that features are ready by backtest start date.

    Returns first valid date where all critical features are available.
    """
    backtest_start_dt = pd.to_datetime(backtest_start)
    critical_features = [
        "yang_zhang_vol_lagged",
        "realized_vol_lagged",
        "har_vol_daily_lagged",
        "rsi_lagged",
        "hurst_lagged",
    ]
    available_features = [f for f in critical_features if f in df.columns]
    if not available_features:
        raise ValueError("No critical features found in DataFrame")
    valid_mask = df[available_features].notna().all(axis=1)
    first_valid = df[valid_mask].index[0] if valid_mask.any() else None
    if first_valid is None:
        raise ValueError("No dates with all critical features available")
    days_lost = (first_valid - backtest_start_dt).days
    print(f"\n[FEATURE WARMUP VALIDATION]")
    print(f"  Backtest start requested: {backtest_start}")
    print(f"  Features ready from: {first_valid.strftime('%Y-%m-%d')}")
    print(f"  Days lost to warmup: {days_lost}")
    if days_lost > 180:
        print(f"  ⚠ WARNING: Losing {days_lost} days ({days_lost / 252:.1f} years)")
        print(f"  Consider reducing warmup_buffer_days or adjusting start_date")
    wf_ready = first_valid + pd.Timedelta(days=config.min_train_window + config.validation_window)
    print(f"  Walk-forward ready: {wf_ready.strftime('%Y-%m-%d')}")
    return (first_valid, wf_ready)


class RegimeDetector:
    """
    Regime detection using XGBoost volatility forecast OR Yang-Zhang.

    FIX V2.4: Use xgb_vol_forecast directly (already uses T-1 features).
    """

    def __init__(
        self,
        config: StrategyConfig,
        vol_lookback: Optional[int] = None,
        quantile_window: Optional[int] = None,
        regime_quantile: Optional[float] = None,
        persistence_days: Optional[int] = None,
    ):
        self.config = config
        self.vol_lookback = vol_lookback or config.vol_lookback
        self.quantile_window = quantile_window or config.quantile_window
        self.regime_quantile = regime_quantile or config.regime_quantile
        self.persistence_days = persistence_days or config.persistence_days

    def detect_regimes(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Detect regimes using XGBoost volatility if available.

        FIX V2.4: Use xgb_vol_forecast (not _lagged) since it already incorporates T-1 data.
        """
        df = df.copy()
        min_xgb_points = min(50, int(len(df) * 0.2))
        if (
            "xgb_vol_forecast" in df.columns
            and df["xgb_vol_forecast"].notna().sum() >= min_xgb_points
        ):
            print(
                f"  [REGIME] Using XGBoost volatility - {df['xgb_vol_forecast'].notna().sum()} predictions"
            )
            df["volatility"] = df["xgb_vol_forecast"].shift(1)
        else:
            xgb_count = (
                df["xgb_vol_forecast"].notna().sum() if "xgb_vol_forecast" in df.columns else 0
            )
            print(
                f"  [REGIME] Using Yang-Zhang volatility (XGBoost: {xgb_count} < {min_xgb_points})"
            )
            df["volatility"] = self._calculate_yz_volatility(df).shift(1)
        df["vol_threshold"] = (
            df["volatility"]
            .rolling(self.quantile_window, closed="left")
            .quantile(self.regime_quantile)
        )
        df["regime_raw"] = (df["volatility"] > df["vol_threshold"]).astype(float)
        df.loc[df["volatility"].isna(), "regime_raw"] = np.nan
        if self.config.use_persistence_filter and self.persistence_days > 1:
            df["regime"] = self._apply_persistence_filter(df["regime_raw"], self.persistence_days)
        else:
            df["regime"] = df["regime_raw"]
        regime_names = {0: "Low Vol", 1: "High Vol"}
        df["regime_name"] = df["regime"].map(regime_names)
        df["regime_duration"] = self._calculate_regime_duration(df["regime"])
        df["returns"] = df["Close"].pct_change()
        df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))
        return df

    def _calculate_yz_volatility(self, df: pd.DataFrame) -> pd.Series:
        """Calculate Yang-Zhang volatility (fallback)."""
        window = self.vol_lookback
        log_open = np.log(df["Open"])
        log_high = np.log(df["High"])
        log_low = np.log(df["Low"])
        log_close = np.log(df["Close"])
        log_close_prev = log_close.shift(1)
        overnight_ret = log_open - log_close_prev
        var_overnight = overnight_ret.rolling(window).var()
        open_close_ret = log_close - log_open
        var_open_close = open_close_ret.rolling(window).var()
        rs_component = (log_high - log_close) * (log_high - log_open) + (log_low - log_close) * (
            log_low - log_open
        )
        var_rs = rs_component.rolling(window).mean()
        k = self.config.yz_k_numerator / (
            self.config.yz_k_denominator_constant + (window + 1) / (window - 1)
        )
        yz_variance = var_overnight + k * var_open_close + (1 - k) * var_rs
        yz_volatility = np.sqrt(yz_variance) * np.sqrt(self.config.yz_annualization_factor)
        return yz_volatility

    def _apply_persistence_filter(self, raw_regime: pd.Series, persistence_days: int) -> pd.Series:
        """Apply persistence filter."""
        filtered = raw_regime.copy()
        valid_mask = ~raw_regime.isna()
        valid_indices = raw_regime[valid_mask].index
        if len(valid_indices) < persistence_days:
            return filtered
        current_regime = raw_regime.loc[valid_indices[0]]
        for i, idx in enumerate(valid_indices):
            raw_value = raw_regime.loc[idx]
            if pd.isna(raw_value):
                filtered.loc[idx] = np.nan
                continue
            if raw_value == current_regime:
                filtered.loc[idx] = current_regime
            else:
                past_count = 0
                for j in range(max(0, i - persistence_days + 1), i):
                    past_idx = valid_indices[j]
                    if raw_regime.loc[past_idx] == raw_value:
                        past_count += 1
                if past_count >= persistence_days - 1:
                    current_regime = raw_value
                    filtered.loc[idx] = current_regime
                else:
                    filtered.loc[idx] = current_regime
        return filtered

    def _calculate_regime_duration(self, regime: pd.Series) -> pd.Series:
        """Calculate regime duration."""
        duration = pd.Series(index=regime.index, dtype=float)
        current_count = 0
        prev_regime = None
        for idx, val in regime.items():
            if pd.isna(val):
                duration.loc[idx] = np.nan
                continue
            if val == prev_regime:
                current_count += 1
            else:
                current_count = 1
                prev_regime = val
            duration.loc[idx] = current_count
        return duration

    def get_signal(self, regime: int) -> float:
        """Convert regime to target allocation."""
        if pd.isna(regime):
            return np.nan
        if regime == 0:
            return self.config.low_vol_allocation
        else:
            return self.config.high_vol_allocation


class TransactionCostCalculator:
    """Calculate realistic transaction costs."""

    def __init__(self, config: StrategyConfig):
        self.config = config

    def calculate_commission(self, shares: float, price: float) -> float:
        """Calculate commission based on IBKR-like structure."""
        shares = abs(shares)
        trade_value = shares * price
        raw_commission = shares * self.config.commission_per_share
        min_commission = self.config.commission_min
        max_commission = trade_value * self.config.commission_max_pct
        return max(min_commission, min(raw_commission, max_commission))

    def calculate_spread_cost(self, shares: float, price: float) -> float:
        """Calculate bid-ask spread cost."""
        trade_value = abs(shares) * price
        return trade_value * (self.config.spread_bps / 10000)

    def calculate_market_impact(self, shares: float, price: float) -> float:
        """Calculate market impact cost."""
        trade_value = abs(shares) * price
        return trade_value * (self.config.market_impact_bps / 10000)

    def calculate_margin_interest(self, borrowed_value: float, days: int = 1) -> float:
        """Calculate margin interest for leveraged positions."""
        if borrowed_value <= 0:
            return 0.0
        daily_rate = self.config.margin_rate_annual / 252
        return abs(borrowed_value) * daily_rate * days

    def calculate_total_cost(self, shares: float, price: float) -> float:
        """Calculate total transaction cost."""
        commission = self.calculate_commission(shares, price)
        spread = self.calculate_spread_cost(shares, price)
        impact = self.calculate_market_impact(shares, price)
        return commission + spread + impact


class BacktestEngine:
    """
    Backtest engine with correct P&L and position sizing.

    FIX V2.4: Implement compounding_mode correctly with cash tracking.
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.cost_calculator = TransactionCostCalculator(config)

    def run_backtest(
        self, df: pd.DataFrame, signals: pd.Series, pre_shifted: bool = False
    ) -> pd.DataFrame:
        """
        Run backtest with hybrid P&L and correct position sizing.

        FIX V2.4: Size on initial_capital when compounding=False, track cash separately.
        """
        df = df.copy()
        if not pre_shifted:
            df["target_allocation"] = signals.shift(1)
        else:
            df["target_allocation"] = signals
        results = []
        portfolio_value = self.config.initial_capital
        current_shares = 0.0
        cash_balance = self.config.initial_capital
        prev_alloc = None
        valid_dates = df.dropna(subset=["Open", "Close"]).index
        for i, date in enumerate(valid_dates):
            open_price = df.loc[date, "Open"]
            close_price = df.loc[date, "Close"]
            target_alloc = df.loc[date, "target_allocation"]
            if i == 0:
                prev_open = open_price
                prev_close = open_price
            else:
                prev_date = valid_dates[i - 1]
                prev_open = df.loc[prev_date, "Open"]
                prev_close = df.loc[prev_date, "Close"]
            if pd.isna(target_alloc):
                if current_shares > 0 and prev_close > 0:
                    holding_ret = close_price / prev_close - 1
                    exposure_prev = current_shares * prev_close / portfolio_value
                    position_ret_pct = exposure_prev * holding_ret
                else:
                    position_ret_pct = 0.0
                position_value = current_shares * close_price
                if self.config.compounding_mode:
                    portfolio_value *= 1 + position_ret_pct
                else:
                    cash_balance = self.config.initial_capital - current_shares * prev_close
                    portfolio_value = position_value + cash_balance
                results.append(
                    {
                        "date": date,
                        "open": open_price,
                        "close": close_price,
                        "target_alloc": np.nan,
                        "shares": current_shares,
                        "position_value": position_value,
                        "cash_balance": cash_balance,
                        "portfolio_value": portfolio_value,
                        "daily_return_pct": position_ret_pct,
                        "transaction_cost": 0.0,
                        "margin_cost": 0.0,
                        "is_execution": False,
                    }
                )
                continue
            signal_changed = prev_alloc is None or abs(target_alloc - prev_alloc) > 0.01
            transaction_cost_pct = 0.0
            margin_cost_pct = 0.0
            if signal_changed:
                if self.config.compounding_mode:
                    sizing_base = portfolio_value
                else:
                    sizing_base = self.config.initial_capital
                target_value = target_alloc * sizing_base * self.config.leverage
                new_shares = target_value / prev_close if prev_close > 0 else 0.0
                shares_traded = new_shares - current_shares
                if abs(shares_traded) > 0.001:
                    cost_dollars = self.cost_calculator.calculate_total_cost(
                        shares_traded, open_price
                    )
                    transaction_cost_pct = cost_dollars / portfolio_value
                if prev_open > 0 and current_shares > 0:
                    market_ret_open = open_price / prev_open - 1
                    old_exposure = current_shares * prev_close / portfolio_value
                    position_ret_pct = old_exposure * market_ret_open
                else:
                    position_ret_pct = 0.0
                position_ret_pct -= transaction_cost_pct
                current_shares = new_shares
                prev_alloc = target_alloc
                is_execution = True
            else:
                if current_shares > 0 and prev_close > 0:
                    market_ret_hold = close_price / prev_close - 1
                    exposure = current_shares * prev_close / portfolio_value
                    position_ret_pct = exposure * market_ret_hold
                else:
                    position_ret_pct = 0.0
                is_execution = False
            position_value_eod = current_shares * close_price
            if position_value_eod > portfolio_value:
                borrowed = position_value_eod - portfolio_value
                margin_cost_dollars = self.cost_calculator.calculate_margin_interest(
                    borrowed, days=1
                )
                margin_cost_pct = margin_cost_dollars / portfolio_value
                position_ret_pct -= margin_cost_pct
            if self.config.compounding_mode:
                portfolio_value *= 1 + position_ret_pct
                cash_balance = portfolio_value - position_value_eod
            else:
                pnl_dollars = portfolio_value * position_ret_pct
                cash_balance += pnl_dollars
                portfolio_value = position_value_eod + cash_balance
            position_value = current_shares * close_price
            results.append(
                {
                    "date": date,
                    "open": open_price,
                    "close": close_price,
                    "target_alloc": target_alloc,
                    "shares": current_shares,
                    "position_value": position_value,
                    "cash_balance": cash_balance,
                    "portfolio_value": portfolio_value,
                    "daily_return_pct": position_ret_pct,
                    "transaction_cost": transaction_cost_pct,
                    "margin_cost": margin_cost_pct,
                    "is_execution": is_execution,
                }
            )
        results_df = pd.DataFrame(results)
        results_df.set_index("date", inplace=True)
        results_df["benchmark_return"] = results_df["close"].pct_change().fillna(0)
        results_df["cumulative_return"] = (
            results_df["portfolio_value"] / self.config.initial_capital - 1
        )
        results_df["cumulative_benchmark"] = (1 + results_df["benchmark_return"]).cumprod() - 1
        results_df["strategy_return"] = results_df["daily_return_pct"]
        return results_df


class PerformanceAnalytics:
    """Calculate performance metrics and robustness statistics."""

    def __init__(self, config: StrategyConfig):
        self.config = config

    def calculate_basic_metrics(
        self,
        returns: pd.Series,
        benchmark_returns: Optional[pd.Series] = None,
        annualization_factor: int = 252,
    ) -> Dict[str, float]:
        """Calculate basic performance metrics."""
        returns = returns.dropna()
        if len(returns) < 20:
            return {
                "error": "Insufficient data",
                "sharpe_ratio": 0,
                "num_observations": len(returns),
            }
        total_return = (1 + returns).prod() - 1
        n_years = len(returns) / annualization_factor
        annual_return = (1 + total_return) ** (1 / n_years) - 1 if n_years > 0 else 0
        volatility = returns.std() * np.sqrt(annualization_factor)
        sharpe = annual_return / volatility if volatility > 0 else 0
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.cummax()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min()
        return {
            "total_return": total_return,
            "annual_return": annual_return,
            "volatility": volatility,
            "sharpe_ratio": sharpe,
            "max_drawdown": max_drawdown,
            "num_observations": len(returns),
        }

    def calculate_deflated_sharpe_ratio(
        self,
        sharpe: float,
        num_observations: int,
        num_trials: int,
        skewness: float = 0,
        kurtosis: float = 3,
    ) -> float:
        """Calculate Deflated Sharpe Ratio (DSR) per Bailey et al."""
        if num_trials <= 1:
            return 0.5
        try:
            euler_mascheroni = 0.5772156649
            z_arg = max(0.0001, min(0.9999, 1 - 1 / num_trials))
            e_max_arg = max(0.0001, min(0.9999, 1 - 1 / (num_trials * np.e)))
            expected_max_sr = (1 - euler_mascheroni) * stats.norm.ppf(
                z_arg
            ) + euler_mascheroni * stats.norm.ppf(e_max_arg)
            sr_std = np.sqrt(
                (1 + 0.5 * sharpe**2 - skewness * sharpe + (kurtosis - 3) / 4 * sharpe**2)
                / num_observations
            )
            expected_max_sr = expected_max_sr * sr_std
            if sr_std > 0:
                z_score = (sharpe - expected_max_sr) / sr_std
                dsr = stats.norm.cdf(z_score)
            else:
                dsr = 0.5
            return max(0, min(1, dsr))
        except:
            return 0.5

    def calculate_probabilistic_sharpe_ratio(
        self,
        sharpe: float,
        num_observations: int,
        benchmark_sharpe: float = 0,
        skewness: float = 0,
        kurtosis: float = 3,
    ) -> float:
        """Calculate Probabilistic Sharpe Ratio (PSR)."""
        if num_observations < 2:
            return 0.5
        variance_term = 1 + 0.5 * sharpe**2 - skewness * sharpe + (kurtosis - 3) / 4 * sharpe**2
        if variance_term <= 0:
            variance_term = 1
        sharpe_std_error = np.sqrt(variance_term / num_observations)
        if sharpe_std_error > 0:
            z_score = (sharpe - benchmark_sharpe) / sharpe_std_error
            psr = stats.norm.cdf(z_score)
        else:
            psr = 0.5
        return max(0, min(1, psr))

    def calculate_global_robustness_index(self, metrics: Dict[str, float]) -> float:
        """Calculate Global Robustness Index (GRI) - combined metric."""
        weights = {"dsr": 0.3, "psr": 0.3, "wfe": 0.4}
        normalized = {}
        normalized["dsr"] = max(0, min(1, metrics.get("dsr", 0.5)))
        normalized["psr"] = max(0, min(1, metrics.get("psr", 0.5)))
        wfe = metrics.get("wfe", 0.5)
        normalized["wfe"] = max(0, min(1, wfe / 1.5))
        return sum((weights[k] * normalized[k] for k in weights))


class HyperparameterOptimizer:
    """Optimize regime detection hyperparameters."""

    def __init__(self, config: StrategyConfig, feature_engineer):
        self.config = config
        self.feature_engineer = feature_engineer
        self.analytics = PerformanceAnalytics(config)

    def generate_param_grid(self) -> List[Dict[str, Any]]:
        """Generate parameter grid for optimization."""
        if self.config.fast_mode:
            vol_grid = [20, 42]
            quant_grid = [252, 504]
            regime_grid = [0.5]
            persist_grid = [3]
        else:
            vol_grid = self.config.vol_lookback_grid
            quant_grid = self.config.quantile_window_grid
            regime_grid = self.config.regime_quantile_grid
            persist_grid = self.config.persistence_days_grid
        grid = list(product(vol_grid, quant_grid, regime_grid, persist_grid))
        return [
            {"vol_lookback": v, "quantile_window": q, "regime_quantile": r, "persistence_days": p}
            for v, q, r, p in grid
        ]

    def optimize(
        self, df: pd.DataFrame, verbose: bool = False
    ) -> Tuple[Dict[str, Any], Dict[str, float]]:
        """Optimize hyperparameters on in-sample data."""
        param_grid = self.generate_param_grid()
        if len(param_grid) > self.config.max_param_combinations:
            np.random.seed(42)
            indices = np.random.choice(
                len(param_grid), self.config.max_param_combinations, replace=False
            )
            param_grid = [param_grid[i] for i in indices]
        if verbose:
            print(f"    Testing {len(param_grid)} parameter combinations...")
        best_objective = -np.inf
        best_params = param_grid[0]
        best_metrics = {}
        for params in param_grid:
            detector = RegimeDetector(
                self.config,
                vol_lookback=params["vol_lookback"],
                quantile_window=params["quantile_window"],
                regime_quantile=params["regime_quantile"],
                persistence_days=params["persistence_days"],
            )
            try:
                regimes = detector.detect_regimes(df)
                signals = regimes["regime"].apply(detector.get_signal)
                valid_signals = signals.dropna()
                if len(valid_signals) < 20:
                    continue
                engine = BacktestEngine(self.config)
                results = engine.run_backtest(regimes, signals, pre_shifted=False)
                returns = results["strategy_return"].dropna()
                if len(returns) < 20:
                    continue
                metrics = self.analytics.calculate_basic_metrics(returns)
                if self.config.optimization_objective == "sharpe":
                    objective = metrics.get("sharpe_ratio", -999)
                else:
                    objective = metrics.get("total_return", -999)
                if np.isnan(objective) or np.isinf(objective):
                    objective = -999
                if objective > best_objective:
                    best_objective = objective
                    best_params = params
                    best_metrics = metrics
            except Exception as e:
                continue
        if best_objective == -np.inf:
            if verbose:
                print("    [WARNING] All parameter combinations failed")
            best_metrics = {"sharpe_ratio": -999, "total_return": 0}
        if verbose:
            print(f"    Best objective: {best_objective:.3f}")
        return (best_params, best_metrics)


class WalkForwardValidator:
    """
    Walk-forward validation with XGBoost retraining each fold.

    FIX: Proper DataFrame updates using .loc to avoid view/copy warnings.
    """

    def __init__(self, config: StrategyConfig, feature_engineer):
        self.config = config
        self.feature_engineer = feature_engineer
        self.analytics = PerformanceAnalytics(config)
        self.optimizer = HyperparameterOptimizer(config, feature_engineer)

    def run_walk_forward(self, df: pd.DataFrame, verbose: bool = True):
        """Run walk-forward validation with XGBoost retraining."""
        if verbose:
            print("\n[WALK-FORWARD] Engineering features WITHOUT XGBoost training...")
        df_features = self.feature_engineer.engineer_features(df, train_xgb=False)
        essential = ["yang_zhang_vol_lagged", "realized_vol_lagged", "returns"]
        valid_data = df_features.dropna(subset=essential)
        valid_indices = valid_data.index
        min_required = self.config.min_train_window + self.config.validation_window
        if len(valid_indices) < min_required:
            raise ValueError(f"Need {min_required} days, got {len(valid_indices)}")
        if verbose:
            print(f"[WALK-FORWARD] Valid data: {len(valid_indices)} days")
        fold_results = []
        param_history = []
        oos_returns_list = []
        fold_num = 0
        train_end_idx = self.config.min_train_window
        while train_end_idx + self.config.validation_window <= len(valid_indices):
            fold_num += 1
            train_indices = valid_indices[:train_end_idx]
            val_indices = valid_indices[
                train_end_idx : train_end_idx + self.config.validation_window
            ]
            if verbose:
                print(
                    f"\nFold {fold_num}: Train {len(train_indices)} days, Val {len(val_indices)} days"
                )
            if self.config.use_xgboost and XGBOOST_AVAILABLE:
                print(f"  Retraining XGBoost on expanding window...")
                train_df_for_xgb = df_features.loc[train_indices]
                xgb_metrics = self.feature_engineer.xgb_predictor.train(
                    train_df_for_xgb, verbose=verbose
                )
                extended_indices = valid_indices[: train_end_idx + self.config.validation_window]
                extended_df = df_features.loc[extended_indices]
                xgb_forecasts = self.feature_engineer.xgb_predictor.predict(extended_df)
                df_features.loc[extended_indices, "xgb_vol_forecast"] = xgb_forecasts.values
                df_features.loc[extended_indices, "xgb_vol_forecast_lagged"] = xgb_forecasts.shift(
                    1
                ).values
                if verbose:
                    valid_preds = xgb_forecasts.notna().sum()
                    print(f"  XGBoost predictions: {valid_preds} / {len(extended_indices)}")
            train_df = df_features.loc[train_indices]
            optimal_params, is_metrics = self.optimizer.optimize(train_df, verbose=verbose)
            param_history.append({"fold": fold_num, **optimal_params})
            if train_end_idx > 0:
                buffer_start_idx = train_end_idx - 1
                extended_indices = valid_indices[
                    buffer_start_idx : train_end_idx + self.config.validation_window
                ]
            else:
                extended_indices = val_indices
            extended_df = df_features.loc[extended_indices]
            detector = RegimeDetector(
                self.config,
                vol_lookback=optimal_params["vol_lookback"],
                quantile_window=optimal_params["quantile_window"],
                regime_quantile=optimal_params["regime_quantile"],
                persistence_days=optimal_params["persistence_days"],
            )
            regimes = detector.detect_regimes(extended_df)
            signals = regimes["regime"].apply(detector.get_signal)
            signals_shifted = signals.shift(1)
            engine = BacktestEngine(self.config)
            results = engine.run_backtest(extended_df, signals_shifted, pre_shifted=True)
            val_results = results.loc[val_indices]
            val_returns = val_results["strategy_return"].dropna()
            if len(val_returns) > 10:
                oos_metrics = self.analytics.calculate_basic_metrics(val_returns)
                oos_sharpe = oos_metrics.get("sharpe_ratio", 0)
            else:
                oos_sharpe = 0.0
            oos_returns_list.append(val_returns)
            is_sharpe = is_metrics.get("sharpe_ratio", 0)
            wfe = oos_sharpe / is_sharpe if abs(is_sharpe) > 0.01 else 0.0
            fold_results.append(
                {"fold": fold_num, "is_sharpe": is_sharpe, "oos_sharpe": oos_sharpe, "wfe": wfe}
            )
            if verbose:
                print(f"  IS: {is_sharpe:.3f}, OOS: {oos_sharpe:.3f}, WFE: {wfe:.3f}")
            train_end_idx += self.config.step_size
        all_oos_returns = pd.concat(oos_returns_list)
        agg_metrics = self.analytics.calculate_basic_metrics(all_oos_returns)
        valid_wfes = [
            f["wfe"] for f in fold_results if not np.isnan(f["wfe"]) and abs(f["wfe"]) < 10
        ]
        avg_wfe = np.mean(valid_wfes) if valid_wfes else 0.0
        dsr = self.analytics.calculate_deflated_sharpe_ratio(
            agg_metrics.get("sharpe_ratio", 0),
            len(all_oos_returns),
            len(self.optimizer.generate_param_grid()) * len(fold_results),
            all_oos_returns.skew(),
            all_oos_returns.kurtosis() + 3,
        )
        psr = self.analytics.calculate_probabilistic_sharpe_ratio(
            agg_metrics.get("sharpe_ratio", 0),
            len(all_oos_returns),
            self.config.psr_benchmark_sharpe,
            all_oos_returns.skew(),
            all_oos_returns.kurtosis() + 3,
        )
        gri_components = {"dsr": dsr, "psr": psr, "wfe": avg_wfe}
        gri = self.analytics.calculate_global_robustness_index(gri_components)
        aggregate = {
            "num_folds": len(fold_results),
            "oos_sharpe": agg_metrics.get("sharpe_ratio", 0),
            "avg_wfe": avg_wfe,
            "dsr": dsr,
            "psr": psr,
            "gri": gri,
            "is_robust": gri > self.config.gri_threshold,
        }
        if verbose:
            print(f"\n[AGGREGATE METRICS]")
            print(f"  OOS Sharpe: {aggregate['oos_sharpe']:.3f}")
            print(f"  DSR: {dsr:.3f}, PSR: {psr:.3f}, WFE: {avg_wfe:.3f}")
            print(f"  GRI: {gri:.3f}")
            print(
                f"  Deployment: {('✓ PROCEED' if aggregate['is_robust'] else '✗ REVIEW REQUIRED')}"
            )
        return {
            "fold_results": fold_results,
            "aggregate_metrics": aggregate,
            "oos_returns": all_oos_returns,
            "param_history": param_history,
            "df_with_features": df_features,
        }


class AntiMonkeyTests:
    """Sanity checks to detect common backtesting errors."""

    @staticmethod
    def test_zero_position_zero_return(results_df, tolerance=1e-06):
        """Zero shares should generate zero returns on holding days."""
        zero_days = results_df[
            (results_df["shares"].abs() < tolerance) & ~results_df["is_execution"]
        ]
        if len(zero_days) == 0:
            return True
        max_ret = zero_days["daily_return_pct"].abs().max()
        if max_ret > 0.01:
            print(f"[FAIL] Zero-share days have return {max_ret:.4f}")
            return False
        return True

    @staticmethod
    def test_position_bounded(results_df, config):
        """Position size should not exceed leverage limit."""
        max_lev = (results_df["position_value"] / results_df["portfolio_value"]).max()
        if max_lev > config.leverage * 1.1:
            print(f"[FAIL] Max leverage {max_lev:.2f} exceeds {config.leverage * 1.1:.2f}")
            return False
        return True

    @staticmethod
    def test_no_lookhead(results_df):
        """First trade should not occur on first day (need lagged signal)."""
        first_trade = results_df[results_df["is_execution"]].index[0]
        if first_trade == results_df.index[0]:
            print("[FAIL] First trade on first day (look-ahead bias)")
            return False
        return True

    @staticmethod
    def test_sharpe_plausible(sharpe, max_sharpe=5.0):
        """Sharpe ratio should be within plausible range."""
        if abs(sharpe) > max_sharpe:
            print(f"[FAIL] Sharpe {sharpe:.2f} exceeds ±{max_sharpe} (likely error)")
            return False
        return True

    @staticmethod
    def run_all_tests(results_df, config, metrics):
        """Run all anti-monkey tests."""
        print("\n[ANTI-MONKEY TESTS]")
        print("-" * 60)
        tests = [
            (
                "Zero position = zero return",
                AntiMonkeyTests.test_zero_position_zero_return(results_df),
            ),
            ("Position bounded", AntiMonkeyTests.test_position_bounded(results_df, config)),
            ("No look-ahead", AntiMonkeyTests.test_no_lookhead(results_df)),
            (
                "Sharpe plausible",
                AntiMonkeyTests.test_sharpe_plausible(metrics.get("sharpe_ratio", 0)),
            ),
        ]
        all_passed = all((passed for _, passed in tests))
        for name, passed in tests:
            print(f"  {('✓ PASS' if passed else '✗ FAIL')}: {name}")
        print("-" * 60)
        return all_passed


def main():
    """Main execution function."""
    print("=" * 80)
    print(" " * 5 + "THRESHOLD REGIME STRATEGY V2.4 - XGBOOST (FIXED)")
    print("=" * 80)
    print(f"Ticker: {TICKER}")
    print(f"XGBoost: {('ENABLED' if USE_XGBOOST and XGBOOST_AVAILABLE else 'DISABLED')}")
    config = StrategyConfig()
    print("\n[STEP 1] Fetching Market Data...")
    start_dt = datetime.strptime(config.start_date, "%Y-%m-%d")
    extended_start = start_dt - timedelta(days=config.warmup_buffer_days)
    fetcher = YahooDataFetcher()
    try:
        df = fetcher.fetch_historical_data(
            config.ticker, extended_start.strftime("%Y-%m-%d"), config.end_date
        )
    except ValueError as e:
        print(f"\n[ERROR] {str(e)}")
        return None
    print("\n[STEP 2] Engineering Features...")
    feature_engineer = EnhancedFeatureEngineer(config)
    df_with_features = feature_engineer.engineer_features(df, train_xgb=False)
    print("\n[STEP 3] Validating Feature Readiness...")
    if config.validate_feature_warmup:
        first_valid, wf_ready = validate_feature_readiness(
            df_with_features, config.start_date, config
        )
        if first_valid > pd.to_datetime(config.start_date):
            print(f"  → Auto-adjusting backtest start to {first_valid.strftime('%Y-%m-%d')}")
            backtest_start = first_valid
        else:
            backtest_start = pd.to_datetime(config.start_date)
    else:
        backtest_start = pd.to_datetime(config.start_date)
    print("\n[STEP 4] Walk-Forward Validation with XGBoost...")
    wf_validator = WalkForwardValidator(config, feature_engineer)
    wf_results = wf_validator.run_walk_forward(df, verbose=True)
    print("\n[STEP 5] XGBoost Leakage Check...")
    if config.use_xgboost and XGBOOST_AVAILABLE:
        feature_importance = feature_engineer.xgb_predictor.get_feature_importance(top_n=30)
        if len(feature_importance) > 0:
            leakage_features = feature_importance[
                feature_importance["feature"].str.contains("xgb_vol", case=False)
            ]
            if len(leakage_features) > 0:
                print(
                    f"  ✗ LEAKAGE DETECTED: {len(leakage_features)} XGBoost self-reference features!"
                )
                print(leakage_features)
            else:
                print(f"  ✓ CLEAN: No XGBoost self-reference in feature set")
        else:
            print(f"  ⚠ No feature importance available")
    print("\n[STEP 6] Running Full Backtest...")
    df_with_features = wf_results["df_with_features"]
    param_history = wf_results["param_history"]
    best_params = (
        param_history[-1]
        if param_history
        else {
            "vol_lookback": config.vol_lookback,
            "quantile_window": config.quantile_window,
            "regime_quantile": config.regime_quantile,
            "persistence_days": config.persistence_days,
        }
    )
    best_params_clean = {k: v for k, v in best_params.items() if k != "fold"}
    final_detector = RegimeDetector(config, **best_params_clean)
    df_with_regimes = final_detector.detect_regimes(df_with_features)
    signals = df_with_regimes["regime"].apply(final_detector.get_signal)
    backtest_engine = BacktestEngine(config)
    backtest_results = backtest_engine.run_backtest(df_with_regimes, signals, pre_shifted=False)
    backtest_end = pd.to_datetime(config.end_date)
    backtest_results = backtest_results[
        (backtest_results.index >= backtest_start) & (backtest_results.index <= backtest_end)
    ]
    final_value = backtest_results["portfolio_value"].iloc[-1]
    total_return = (final_value / config.initial_capital - 1) * 100
    num_trades = backtest_results["is_execution"].sum()
    analytics = PerformanceAnalytics(config)
    final_metrics = analytics.calculate_basic_metrics(backtest_results["strategy_return"].dropna())
    print("\n[STEP 7] Running Anti-Monkey Tests...")
    tests_passed = AntiMonkeyTests.run_all_tests(backtest_results, config, final_metrics)
    print("\n[STEP 8] Exporting Results...")
    output_base = f"results/google-volatility-regimes-extended/{TICKER.lower()}"
    backtest_results.to_csv(f"{output_base}_backtest.csv")
    pd.DataFrame(wf_results["fold_results"]).to_csv(f"{output_base}_walkforward.csv", index=False)
    pd.DataFrame(param_history).to_csv(f"{output_base}_params.csv", index=False)
    if config.use_xgboost and XGBOOST_AVAILABLE:
        feature_importance = feature_engineer.xgb_predictor.get_feature_importance(top_n=30)
        if len(feature_importance) > 0:
            feature_importance.to_csv(f"{output_base}_xgb_importance.csv", index=False)
            print(f"\nTop 10 XGBoost Features:")
            print(feature_importance.head(10).to_string(index=False))
    agg_metrics = wf_results["aggregate_metrics"].copy()
    agg_metrics["anti_monkey_passed"] = tests_passed
    agg_metrics["xgboost_used"] = config.use_xgboost and XGBOOST_AVAILABLE
    with open(f"{output_base}_metrics.json", "w") as f:
        json.dump(agg_metrics, f, indent=2, default=str)
    print(f"\n  Files saved: {output_base}_*")
    print("\n" + "=" * 80)
    print("EXECUTION COMPLETE")
    print("=" * 80)
    print(f"\nFinal Value: ${final_value:,.2f}")
    print(f"Total Return: {total_return:.2f}%")
    print(f"Sharpe Ratio: {final_metrics.get('sharpe_ratio', 0):.3f}")
    print(f"Max Drawdown: {final_metrics.get('max_drawdown', 0) * 100:.2f}%")
    print(f"Number of Trades: {num_trades}")
    print(f"\nOOS Sharpe: {agg_metrics['oos_sharpe']:.3f}")
    print(f"GRI: {agg_metrics['gri']:.3f}")
    print(f"\nAnti-Monkey Tests: {('✓ PASS' if tests_passed else '✗ FAIL')}")
    print(f"Robustness: {('✓ PASS' if agg_metrics['is_robust'] else '✗ REVIEW REQUIRED')}")
    deployment_status = (
        "PROCEED" if tests_passed and agg_metrics["is_robust"] else "REQUIRES REVIEW"
    )
    print(f"\nDeployment Status: {deployment_status}")
    return {
        "config": config,
        "backtest_results": backtest_results,
        "wf_results": wf_results,
        "tests_passed": tests_passed,
    }


if __name__ == "__main__":
    results = main()
